# Atualizando definições de tools dos seus MCP servers no AgentCore Gateway

## Visão Geral

O mecanismo de sincronização no AgentCore Gateway garante definições precisas de tools a partir de MCP server targets. Ele gerencia a consistência de schema, otimiza o desempenho e mantém a integridade dos dados por meio de métodos de sincronização explícita e implícita.

### Sincronização Explícita

A API SynchronizeGateway é uma operação assíncrona que permite a sincronização sob demanda de tools a partir de MCP server targets. Esta API fornece aos clientes controle explícito sobre quando atualizar suas definições de tools, sendo particularmente útil após fazer alterações nas configurações de tools do seu MCP server. Para targets configurados com autenticação OAuth, a API primeiro valida as credenciais por meio do serviço AgentCore Identity antes de prosseguir com a comunicação com o MCP server. Se a validação falhar, a operação de sincronização falha com detalhes de erro apropriados, transicionando o target para um estado FAILED. Para targets configurados sem autenticação, a API prossegue diretamente para a sincronização de tools.

O fluxo de trabalho de processamento de tools começa com o estabelecimento de uma sessão com o MCP server. A API então recupera e processa tools em lotes otimizados, adicionando prefixos específicos do target para evitar colisões de nomes com tools de outros targets. A API impõe um limite de 10.000 tools por target e garante que as definições de tools sejam normalizadas, preservando metadados essenciais das definições originais do MCP server.

Neste tutorial, vamos atualizar o MCP Server criado em '01-Gateway-MCP-Target' adicionando tools adicionais e então invocar a API SynchronizeGateway para sincronizar explicitamente as definições de tools mais recentes.

![Diagram](images/mcp-server-target-explicit-sync.png)

### Sincronização Implícita

O AgentCore Gateway realiza uma sincronização implícita durante as operações CreateGatewayTarget e UpdateGatewayTarget que difere da API explícita SynchronizeGateway. Esta sincronização integrada garante que MCP targets recém-criados ou atualizados sejam imediatamente utilizáveis, mantendo a consistência dos dados. Esta sincronização implícita garante que MCP targets sejam sempre criados ou atualizados com definições de tools válidas e atuais, mantendo a garantia do Gateway de que qualquer target no estado READY esteja imediatamente utilizável. 

Neste tutorial, vamos criar um novo MCP Server e chamar a operação UpdateGatewayTarget para atualizar o novo MCP target que sincroniza implicitamente as definições de tools do novo MCP Server.

![Diagram](images/mcp-server-target-implicit-sync.png)

### Detalhes do Tutorial


| Informação           | Detalhes                                                                         |
|:---------------------|:---------------------------------------------------------------------------------|
| Tipo do tutorial     | Interativo                                                                       |
| Componentes AgentCore| AgentCore Gateway, AgentCore Identity, AgentCore Runtime                         |
| Framework de Agentes | Strands Agents                                                                   |
| Tipo de Gateway Target| MCP server                                                                      |
| Agente               | Strands                                                                          |
| Inbound Auth IdP| Amazon Cognito, mas pode usar outros                                            |
| Outbound Auth        | Amazon Cognito, mas pode usar outros                                             |
| Modelo LLM           | Anthropic Claude Sonnet 4                                                        |
| Componentes do tutorial| Criando AgentCore Gateway com MCP Target e sincronizando as tools               |
| Vertical do tutorial | Cross-vertical                                                                   |
| Complexidade do exemplo| Fácil                                                                           |
| SDK utilizado        | boto3                                                                            |

## Arquitetura do Tutorial

Neste tutorial, vamos descrever como acionar tanto a sincronização explícita quanto a implícita de definições de tools a partir de MCP server targets dentro do AgentCore Gateway

## Pré-requisitos

Para executar este tutorial você precisará de:
* Jupyter notebook (kernel Python)
* uv
* Credenciais AWS
* Amazon Cognito

In [ ]:
# Install from the requirements file or pyproject.toml file in current directory
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
# Set AWS credentials if not using Amazon SageMaker notebook
import os
# os.environ['AWS_ACCESS_KEY_ID'] = '' # Set the access key
# os.environ['AWS_SECRET_ACCESS_KEY'] = '' # Set the secret key
os.environ['AWS_DEFAULT_REGION'] = os.environ.get('AWS_REGION', 'us-east-1')

In [ ]:
# Import utils
import os
import sys

# Get the directory of the current script
if '__file__' in globals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.getcwd()  # Fallback if __file__ is not defined (e.g., Jupyter)

# Navigate to the directory containing utils.py (one level up)
utils_dir = os.path.abspath(os.path.join(current_dir, '..'))

# Add to sys.path
sys.path.insert(0, utils_dir)

# Now you can import utils
import utils

# Setup logging 
import logging

# Configure logging for notebook environment
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler()]
)

# Set specific logger levels
logging.getLogger("gateway").setLevel(logging.INFO)

# Realizar Sincronização Explícita

## Recuperar as Informações do Cognito User Pool para inbound auth no Gateway

In [ ]:
# Creating Cognito User Pool 
import os
import boto3

REGION = os.environ['AWS_DEFAULT_REGION']
USER_POOL_NAME = "sample-agentcore-gateway-pool"
RESOURCE_SERVER_ID = "sample-agentcore-gateway-id"
RESOURCE_SERVER_NAME = "sample-agentcore-gateway-name"
CLIENT_NAME = "sample-agentcore-gateway-client"
SCOPES = [
    {"ScopeName": "invoke",  # Just 'invoke', will be formatted as resource_server_id/invoke
    "ScopeDescription": "Scope for invoking the agentcore gateway"},
]

scope_names = [f"{RESOURCE_SERVER_ID}/{scope['ScopeName']}" for scope in SCOPES]
scopeString = " ".join(scope_names)


cognito = boto3.client("cognito-idp", region_name=REGION)

print("Creating or retrieving Cognito resources...")
gw_user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"User Pool ID: {gw_user_pool_id}")

utils.get_or_create_resource_server(cognito, gw_user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES)
print("Resource server ensured.")

gw_client_id, gw_client_secret = utils.get_or_create_m2m_client(cognito, gw_user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID, scope_names)

print(f"Client ID: {gw_client_id}")

# Get discovery URL
gw_cognito_discovery_url = f'https://cognito-idp.{REGION}.amazonaws.com/{gw_user_pool_id}/.well-known/openid-configuration'
print(gw_cognito_discovery_url)

## Recuperar as Informações do Cognito User Pool para inbound auth no Runtime

In [ ]:
# Creating Cognito User Pool
import os
import boto3

REGION = os.environ['AWS_DEFAULT_REGION']
USER_POOL_NAME = "sample-agentcore-runtime-pool"
RESOURCE_SERVER_ID = "sample-agentcore-runtime-id"
RESOURCE_SERVER_NAME = "sample-agentcore-runtime-name"
CLIENT_NAME = "sample-agentcore-runtime-client"
SCOPES = [
    {"ScopeName": "invoke",  # Just 'invoke', will be formatted as resource_server_id/invoke
    "ScopeDescription": "Scope for invoking the agentcore gateway"},
]

scope_names = [f"{RESOURCE_SERVER_ID}/{scope['ScopeName']}" for scope in SCOPES]
runtimeScopeString = " ".join(scope_names)


cognito = boto3.client("cognito-idp", region_name=REGION)

print("Creating or retrieving Cognito resources...")
runtime_user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"User Pool ID: {runtime_user_pool_id}")

utils.get_or_create_resource_server(cognito, runtime_user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES)
print("Resource server ensured.")

runtime_client_id, runtime_client_secret = utils.get_or_create_m2m_client(cognito, runtime_user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID, scope_names)

print(f"Client ID: {runtime_client_id}")

# Get discovery URL
runtime_cognito_discovery_url = f'https://cognito-idp.{REGION}.amazonaws.com/{runtime_user_pool_id}/.well-known/openid-configuration'
print(runtime_cognito_discovery_url)

## Adicionar uma nova tool ao MCP Server existente

Adicione uma nova tool 'cancelOrder' ao MCP Server existente que você criou no tutorial '01-mcp-server-target'.

In [ ]:
%%writefile mcp_server_updated.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP(host="0.0.0.0", stateless_http=True)

@mcp.tool()
def getOrder() -> int:
    """Get an order"""
    return 123

@mcp.tool()
def updateOrder(orderId: int) -> int:
    """Update existing order"""
    return 456

@mcp.tool()
def cancelOrder(orderId: int) -> int:
    """cancel existing order"""
    return 789

if __name__ == "__main__":
    mcp.run(transport="streamable-http")

Em seguida, vamos configurar o AgentCore Runtime para implantar o MCP server atualizado com a nova tool 'cancelOrder'.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
print(f"Using AWS region: {region}")

required_files = ['mcp_server_updated.py', 'requirements.txt']
for file in required_files:
    if not os.path.exists(file):
        raise FileNotFoundError(f"Required file {file} not found")
print("All required files found ✓")
agentcore_runtime = Runtime()

auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [
            runtime_client_id
        ],
        "discoveryUrl": runtime_cognito_discovery_url
    }
}

print("Configuring AgentCore Runtime...")
response = agentcore_runtime.configure(
    entrypoint="mcp_server_updated.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    authorizer_configuration=auth_config,
    protocol="MCP",
    agent_name="ac_gateway_mcp_server"
)
print("Configuration completed ✓")

Então, reimplante o MCP Server atualizado no AgentCore Runtime.

In [ ]:
print("Launching MCP server to AgentCore Runtime...")
print("This may take several minutes...")
launch_result = agentcore_runtime.launch()

agent_arn = launch_result.agent_arn
agent_id = launch_result.agent_id

encoded_arn = agent_arn.replace(':', '%3A').replace('/', '%2F')

agent_url = f'https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT'
print("Launch completed ✓")
print(f"Agent ARN: {agent_arn}")
print(f"Agent ID: {agent_id}")

## Invocar a API SynchronizeGateway

### Recuperar o provedor de Outbound Auth

Recupere o provedor de credenciais OAuth2 existente que criamos no notebook anterior

In [ ]:
identity_client = boto3.client('bedrock-agentcore-control', region_name=region)

# Get credential provider 
response = identity_client.get_oauth2_credential_provider(name="ac-gateway-mcp-server-identity")
cognito_provider_arn = response['credentialProviderArn']
print(cognito_provider_arn)

### Verificar as definições de tools atuais

Vamos invocar um agente strands para listar as MCP tools usando o Bedrock AgentCore Gateway. Isso retornará a lista antiga de tools (sem 'cancelOrder'), porque ainda não pedimos ao Gateway para sincronizar.

Recupere o identificador do gateway que criamos no notebook anterior.

In [ ]:
gateway_name = 'ac-gateway-mcp-server'

gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
list_gateways_response = gateway_client.list_gateways()

gatewayID = next(
    g['gatewayId'] for g in list_gateways_response['items'] 
    if g['name'] == gateway_name
)

print(gatewayID)

Recupere a URL do gateway.

In [ ]:
gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
gateway_response = gateway_client.get_gateway(gatewayIdentifier=gatewayID)

gatewayURL = gateway_response['gatewayUrl']

print(gatewayURL)

In [ ]:
import json

import requests
from strands.models import BedrockModel
from mcp.client.streamable_http import streamablehttp_client
from strands.tools.mcp.mcp_client import MCPClient
from strands import Agent


def get_token():
    token = utils.get_token(gw_user_pool_id, gw_client_id, gw_client_secret, scopeString, REGION)
    return token['access_token']


def create_streamable_http_transport():
    return streamablehttp_client(
        gatewayURL, headers={"Authorization": f"Bearer {get_token()}"}
    )


client = MCPClient(create_streamable_http_transport)

## The IAM group/user/ configured in ~/.aws/credentials should have access to Bedrock model
yourmodel = BedrockModel(
    model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0", # may need to update model_id depending on region
    temperature=0.7,
    max_tokens=500,  # Limit response length
)

with client:
    # Call the listTools
    tools = client.list_tools_sync()
    # Create an Agent with the model and tools
    agent = Agent(
        model=yourmodel, tools=tools
    )  ## you can replace with any model you like
    # Invoke the agent with the sample prompt. This will only invoke MCP listTools and retrieve the list of tools the LLM has access to. The below does not actually call any tool.
    agent("Hi, can you list all tools available to you")  

### Chamar a API SynchronizeGateway 

Recupere o Id do MCP Target

In [ ]:
gateway_target_name = 'mcp-server-target'

gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
list_targets_response = gateway_client.list_gateway_targets(gatewayIdentifier=gatewayID)

gatewayTargetID = next(
    g['targetId'] for g in list_targets_response['items'] 
    if g['name'] == gateway_target_name
)

print(gatewayTargetID)

Chame a API Synchronize Gateway para realizar a sincronização explícita

In [ ]:
gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)

synchronize_gateway_response = gateway_client.synchronize_gateway_targets(
    gatewayIdentifier=gatewayID,
    targetIdList=[gatewayTargetID]
    )

print(synchronize_gateway_response)

### Verificar as definições de tools atuais após a sincronização

Vamos invocar um agente strands para listar as MCP tools usando o Bedrock AgentCore Gateway. Isso retornará a nova lista de tools.

In [ ]:
with client:
    # Call the listTools
    tools = client.list_tools_sync()
    # Create an Agent with the model and tools
    agent = Agent(
        model=yourmodel, tools=tools
    )  ## you can replace with any model you like
    # Invoke the agent with the sample prompt. This will only invoke MCP listTools and retrieve the list of tools the LLM has access to. The below does not actually call any tool.
    agent("Hi, can you list all tools available to you")   

# Realizar Sincronização Implícita

## Adicionar outra nova tool ao MCP Server existente

Adicione uma nova tool 'deleteOrder' ao MCP Server existente.

In [ ]:
%%writefile mcp_server_updated.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP(host="0.0.0.0", stateless_http=True)

@mcp.tool()
def getOrder() -> int:
    """Get an order"""
    return 123

@mcp.tool()
def updateOrder(orderId: int) -> int:
    """Update existing order"""
    return 456

@mcp.tool()
def cancelOrder(orderId: int) -> int:
    """cancel existing order"""
    return 789

@mcp.tool()
def deleteOrder(orderId: int) -> int:
    """delete existing order"""
    return 101
    
if __name__ == "__main__":
    mcp.run(transport="streamable-http")

Em seguida, vamos configurar o AgentCore Runtime para implantar o MCP server atualizado com a nova tool 'deleteOrder'.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
print(f"Using AWS region: {region}")

required_files = ['mcp_server_updated.py', 'requirements.txt']
for file in required_files:
    if not os.path.exists(file):
        raise FileNotFoundError(f"Required file {file} not found")
print("All required files found ✓")
agentcore_runtime = Runtime()

auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [
            runtime_client_id
        ],
        "discoveryUrl": runtime_cognito_discovery_url
    }
}

print("Configuring AgentCore Runtime...")
response = agentcore_runtime.configure(
    entrypoint="mcp_server_updated.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    authorizer_configuration=auth_config,
    protocol="MCP",
    agent_name="ac_gateway_mcp_server"
)
print("Configuration completed ✓")

Então, reimplante o MCP Server atualizado no AgentCore Runtime.

In [ ]:
print("Launching MCP server to AgentCore Runtime...")
print("This may take several minutes...")
launch_result = agentcore_runtime.launch()

agent_arn = launch_result.agent_arn
agent_id = launch_result.agent_id

encoded_arn = agent_arn.replace(':', '%3A').replace('/', '%2F')

agent_url = f'https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT'
print("Launch completed ✓")
print(f"New Agent ARN: {agent_arn}")
print(f"New Agent ID: {agent_id}")

## Atualizar o Gateway Target existente com o MCP Server

Esta operação de atualização do gateway target atualizará o gateway target com o MCP Server (nova versão) e também realizará a sincronização implícita para sincronizar as definições de tools

In [ ]:
import boto3

gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
update_gateway_target_response = gateway_client.update_gateway_target(
    gatewayIdentifier=gatewayID,
    targetId = gatewayTargetID,
    name = 'mcp-server-target',
    targetConfiguration={
        'mcp': {
            'mcpServer': {
                'endpoint': agent_url
            }
        }
    },
    credentialProviderConfigurations=[
        {
            'credentialProviderType': 'OAUTH',
            'credentialProvider': {
                'oauthCredentialProvider': {
                    'providerArn': cognito_provider_arn,
                    'scopes': [
                        runtimeScopeString
                    ]
                }
            }
        },
    ]
)

print(update_gateway_target_response)

## Verificar as definições de tools atuais após atualizar o MCP Server target

In [ ]:
with client:
    # Call the listTools
    tools = client.list_tools_sync()
    # Create an Agent with the model and tools
    agent = Agent(
        model=yourmodel, tools=tools
    )  ## you can replace with any model you like
    # Invoke the agent with the sample prompt. This will only invoke MCP listTools and retrieve the list of tools the LLM has access to. The below does not actually call any tool.
    agent("Hi, can you list all tools available to you")   

# Limpeza
Recursos adicionais também são criados, como IAM role, IAM Policies, provedor de credenciais, funções AWS Lambda, Cognito user pools, buckets s3 que você pode precisar excluir manualmente como parte da limpeza. Isso depende do exemplo que você executar.

## Excluir o Gateway (Opcional)

In [ ]:
import boto3
gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)

utils.delete_gateway(gateway_client, gatewayID)

### Excluir o Identity Provider (Opcional)

In [ ]:
identity_client.delete_oauth2_credential_provider(name='ac-gateway-mcp-server-identity')


#### Excluir o Runtime (Opcional) 

In [ ]:
import boto3
runtime_client = boto3.client('bedrock-agentcore-control',
                              region_name=REGION)

runtime_client.delete_agent_runtime(agentRuntimeId=agent_id)